In [6]:
#INSTALACE BALÍČKŮ
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
#https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species
import pandas as pd

In [7]:
#INICIALIZACE MODELU
model_name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"
#případně menší model - zkontroluj dokumentaci

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForMaskedLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

model.eval()

c:\Users\sztac\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: c40b5bc7-271c-4c7b-876e-67d887d6343b)')' thrown while requesting HEAD https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(4107, 1024, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(2050, 1024, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-28): 29 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1024,), eps=1e-12

In [8]:
#ZADÁNÍ SEKVENCE
sequence = "ATGCGTACGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG"
#zadáš si vlastní sekvence

breakpoint = len(sequence) // 2 

In [9]:
#ZÍSKÁNÍ EMBEDDINGŮ
tokens = tokenizer(
    sequence,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=tokenizer.model_max_length
)
#pro následné anlýzy je nutné mít všechny sekvence stejně dlouhé

with torch.no_grad():
    outputs = model(
        tokens["input_ids"],
        attention_mask=tokens["attention_mask"],
        output_hidden_states=True
    )

embeddings = outputs.hidden_states[-1]

breakpoint_token = breakpoint + 1

breakpoint_embedding = embeddings[0, breakpoint_token, :]

In [10]:
#ULOŽENÍ DO SOUBORU
embedding_np = breakpoint_embedding.detach().cpu().numpy()

df = pd.DataFrame(embedding_np.reshape(1, -1))

df.to_csv("breakpoint_embedding.csv", index=False)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
#https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-500m-multi-species
import pandas as pd
%pip uninstall -y transformers accelerate

# Instalace stabilní kombinace pro Nucleotide Transformer
%pip install transformers==4.40.0 accelerate==0.29.0

c:\Users\sztac\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: accelerate 0.29.0
Uninstalling accelerate-0.29.0:
  Successfully uninstalled accelerate-0.29.0
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached accelerate-0.29.0-py3-none-any.whl.metadata (18 kB)
Using cached transformers-4.40.0-py3-none-any.whl (9.0 MB)
Using cached accelerate-0.29.0-py3-none-any.whl (297 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
#Grafika reinstal
import sys
import subprocess

def install_gpu_torch():
    print("--- ZAČÍNÁM OPRAVU PRO GPU (RTX 3060) ---")
    
    # 1. Odinstalace starého Torche
    print("\nKrok 1: Mažu starou verzi (CPU)...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"])
    
    # 2. Instalace správné verze pro tvoji grafiku (CUDA 12.1)
    # Pozor: Tohle stahuje cca 2.5 GB, tak to chvíli potrvá!
    print("\nKrok 2: Stahuji verzi pro tvoji grafiku (cca 2.5 GB)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", 
        "torch", "torchvision", "torchaudio", 
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ])
    
    print("\n" + "="*40)
    print("HOTOVO! TEĎ JE TO KRITICKÉ:")
    print("1. Podívej se nahoru do lišty VS Code nad tímto kódem.")
    print("2. Klikni na tlačítko 'Restart' nebo 'Restart Kernel' (točící se šipka).")
    print("3. Teprve pak zkus spustit kód s modelem.")
    print("="*40)

install_gpu_torch()


--- ZAČÍNÁM OPRAVU PRO GPU (RTX 3060) ---

Krok 1: Mažu starou verzi (CPU)...

Krok 2: Stahuji verzi pro tvoji grafiku (cca 2.5 GB)...

HOTOVO! TEĎ JE TO KRITICKÉ:
1. Podívej se nahoru do lišty VS Code nad tímto kódem.
2. Klikni na tlačítko 'Restart' nebo 'Restart Kernel' (točící se šipka).
3. Teprve pak zkus spustit kód s modelem.


In [ ]:
#grafika test
import torch
if torch.cuda.is_available():
    print(f"ÚSPĚCH! Grafika {torch.cuda.get_device_name(0)} je připravena k boji.")
else:
    print("Něco je špatně, stále jedu na CPU.")

ÚSPĚCH! Grafika NVIDIA GeForce RTX 3060 Laptop GPU je připravena k boji.


In [18]:
#START
import torch
import pandas as pd
import numpy as np
import sys
from transformers import AutoTokenizer, AutoModel  # Tohle ti chybělo!
from torch.nn.functional import cosine_similarity

# Nastavení zařízení na GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám: {device}")
if torch.cuda.is_available():
    print(f"Grafická karta: {torch.cuda.get_device_name(0)}")

from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

model_name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Načtení tokenizeru
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Načtení modelu v plné přesnosti (float32)
model = AutoModelForMaskedLM.from_pretrained(
    model_name, 
    trust_remote_code=True
    # torch_dtype je pryč, default je float32
)

model = model.to(device)
model.eval()

print(f"Model je načten na zařízení: {device}")

Používám: cuda
Grafická karta: NVIDIA GeForce RTX 3060 Laptop GPU
Model je načten na zařízení: cuda


In [20]:
#Nacteni sekvence a BP
full_sequence = """TTCCAGGACTGCAGAACTGGCCCAGACCTCTGTATTGGAAAGGTCTTTATGGACCAGGGAGTCCGGTGTCTTTTTTACGGGGGACCCCTG
GGCTGCGAGTTGCACAGTCCAATTCGCTGTTGTTAGGGCCTCAGTTTCCCAAAAGGCACAGGGACGGGGGGAGGGTGGCGGCTCGATGGG
GGAGCCGCCTCCAGGGGGCCCCCCCGCCCTGTGCCCACGGCGCGGCCCCTTTAAGAGGCCCGCCTGGCTCCGTCATCCGCGCCGCGGCCA
CCTCCCCCCGGCCCTCCCCTTCCTGCGGCGCAGAGTGCGGGCCGGGCGGGAGTGCGGCGAGAGCCGGCTGGCTGAGCTTAGCGTCCGAGG
AGGCGGCGGCGGCGGCGGCGGCACGGCGGCGGCGGGGCTGTGGGGCGGTGCGGAAGCGAGAGGCGAGGAGCGCGCGGGCCGTGGCCAGAG
TCTGGCGGCGGCCTGGCGGAGCGGAGAGCAGCGCCCGCGCCTCGCCGTGCGGAGGAGCCCCGCACACAATAGCGGCGCGCGCAGCCCGCG
CCCTTCCCCCCGGCGCGCCCCGCCCCGCGCGCCGAGCGCCCCGCTCCGCCTCACCTGCCACCAGGGAGTGGGCGGGCATTGTTCGCCGCC
GCCGCCGCCGCGCGGGCCATGGGGGCCGCCCGGCGCCCGGGGCCGGGCTGGCGAGGCGCCGCGCCGCCGCTGAGACGGGCCCCGCGCGCA
GCCCGGCGGCGCAGGTAAGGCCGGCCGCGCCATGGTGGACCCGGTGGGCTTCGCGGAGGCGTGGAAGGCGCAGTTCCCGGACTCAGAGCC
CCCGCGCATGGAGCTGCGCTCAGTGGGCGACATCGAGCAGGAGCTGGAGCGCTGCAAGGCCTCCATTCGGCGCCTGGAGCAGGAGGTGAA
CCAGGAGCGCTTCCGCATGATCTACCTGCAGACGTTGCTGGCCAAGGAAAAGAAGAGCTATGACCGGCAGCGATGGGGCTTCCGGCGCGC
GGCGCAGGCCCCCGACGGCGCCTCCGAGCCCCGAGCGTCCGCGTCGCGCCCGCAGCCAGCGCCCGCCGACGGAGCCGACCCGCCGCCCGC
CGAGGAGCCCGAGGCCCGGCCCGACGGCGAGGGTTCTCCGGGTAAGGCCAGGCCCGGGACCGCCCGCAGGCCCGGGGCAGCCGCGTCGGG
GGAACGGGACGACCGGGGACCCCCCGCCAGCGTGGCGGCGCTCAGGTCCAACTTCGAGCGGATCCGCAAGGGCCATGGCCAGCCCGGGGC
GGACGCCGAGAAGCCCTTCTACGTGAACGTCGAGTTTCACCACGAGCGCGGCCTGGTGAAGGTCAACGACAAAGAGGTGTCGGACCGCAT
CAGCTCCCTGGGCAGCCAGGCCATGCAGATGGAGCGCAAAAAGTCCCAGCACGGCGCGGGCTCGAGCGTGGGGGATGCATCCAGGCCCCC
TTACCGGGGACGCTCCTCGGAGAGCAGCTGCGGCGTCGACGGCGACTACGAGGACGCCGAGTTGAACCCCCGCTTCCTGAAGGACAACCT
GATCGACGCCAATGGCGGTAGCAGGCCCCCTTGGCCGCCCCTGGAGTACCAGCCCTACCAGAGCATCTACGTCGGGGGCATGATGGAAGG
GGAGGGCAAGGGCCCGCTCCTGCGCAGCCAGAGCACCTCTGAGCAGGAGAAGCGCCTTACCTGGCCCCGCAGGTCCTACTCCCCCCGGAG
TTTTGAGGATTGCGGAGGCGGCTATACCCCGGACTGCAGCTCCAATGAGAACCTCACCTCCAGCGAGGAGGACTTCTCCTCTGGCCAGTC
CAGCCGCGTGTCCCCAAGCCCCACCACCTACCGCATGTTCCGGGACAAAAGCCGCTCTCCCTCGCAGAACTCGCAACAGTCCTTCGACAG
CAGCAGTCCCCCCACGCCGCAGTGCCATAAGCGGCACCGGCACTGCCCGGTTGTCGTGTCCGAGGCCACCATCGTGGGCGTCCGCAAGAC
CGGGCAGATCTGGCCCAACGATGGCGAGGGCGCCTTCCATGGAGACGCAGATGGCTCGTTCGGAACACCACCTGGATACGGCTGCGCTGC
AGACCGGGCAGAGGAGCAGCGCCGGCACCAAGATGGGCTGCCCTACATTGATGACTCGCCCTCCTCATCGCCCCACCTCAGCAGCAAGGG
CAGGGGCAGCCGGGATGCGCTGGTCTCGGGAGCCCTGGAGTCCACTAAAGCGAGTGAGCTGGACTTGGAAAAGGGCTTGGAGATGAGAAA
ATGGGTCCTGTCGGGAATCCTGGCTAGCGAGGAGACTTACCTGAGCCACCTGGAGGCACTGCTGCTGCCCATGAAGCCTTTGAAAGCCGC
TGCCACCACCTCTCAGCCGGTGCTGACGAGTCAGCAGATCGAGACCATCTTCTTCAAAGTGCCTGAGCTCTACGAGATCCACAAGGAGTT
CTATGATGGGCTCTTCCCCCGCGTGCAGCAGTGGAGCCACCAGCAGCGGGTGGGCGACCTCTTCCAGAAGCTGGCCAGCCAGCTGGGTGT
GTACCGGGCCTTCGTGGACAACTACGGAGTTGCCATGGAAATGGCTGAGAAGTGCTGTCAGGCCAATGCTCAGTTTGCAGAAATCTCCGA
GAACCTGAGAGCCAGAAGCAACAAAGATGCCAAGGATCCAACGACCAAGAACTCTCTGGAAACTCTGCTCTACAAGCCTGTGGACCGTGT
GACGAGGAGCACGCTGGTCCTCCATGACTTGCTGAAGCACACTCCTGCCAGCCACCCTGACCACCCCTTGCTGCAGGACGCCCTCCGCAT
CTCACAGAACTTCCTGTCCAGCATCAATGAGGAGATCACACCCCGACGGCAGTCCATGACGGTGAAGAAGGGAGAGCACCGGCAGCTGCT
GAAGGACAGCTTCATGGTGGAGCTGGTGGAGGGGGCCCGCAAGCTGCGCCACGTCTTCCTGTTCACCGACCTGCTTCTCTGCACCAAGCT
CAAGAAGCAGAGCGGAGGCAAAACGCAGCAGTATGACTGCAAATGGTACATTCCGCTCACGGATCTCAGCTTCCAGATGGTGGATGAACT
GGAGGCAGTGCCCAACATCCCCCTGGTGCCCGATGAGGAGCTGGACGCTTTGAAGATCAAGATCTCCCAGATCAAGAATGACATCCAGAG
AGAGAAGAGGGCGAACAAGGGCAGCAAGGCTACGGAGAGGCTGAAGAAGAAGCTGTCGGAGCAGGAGTCACTGCTGCTGCTTATGTCTCC
CAGCATGGCCTTCAGGGTGCACAGCCGCAACGGCAAGAGTTACACGTTCCTGATCTCCTCTGACTATGAGCGTGCAGAGTGGAGGGAGAA
CATCCGGGAGCAGCAGAAGAAGTGTTTCAGAAGCTTCTCCCTGACATCCGTGGAGCTGCAGATGCTGACCAACTCGTGTGTGAAACTCCA
GACTGTCCACAGCATTCCGCTGACCATCAATAAGGAAGATGATGAGTCTCCGGGGCTCTATGGGTTTCTGAATGTCATCGTCCACTCAGC
CACTGGATTTAAGCAGAGTTCAAATCTGTACTGCACCCTGGAGGTGGATTCCTTTGGGTATTTTGTGAATAAAGCAAAGACGCGCGTCTA
CAGGGACACAGCTGAGCCAAACTGGAACGAGGAATTTGAGATAGAGCTGGAGGGCTCCCAGACCCTGAGGATACTGTGCTATGAAAAGTG
TTACAACAAGACGAAGATCCCCAAGGAGGACGGCGAGAGCACGGACAGACTCATGGGGAAGGGCCAGGTCCAGCTGGACCCGCAGGCCCT
GCAGGACAGAGACTGGCAGCGCACCGTCATCGCCATGAATGGGATCGAAGTAAAGCTCTCGGTCAAGTTCAACAGCAGGGAGTTCAGCTT
GAAGAGGATGCCGTCCCGAAAACAGACAGGGGTCTTCGGAGTCAAGATTGCTGTGGTCACCAAGAGAGAGAGGTCCAAGGTGCCCTACAT
CGTGCGCCAGTGCGTGGAGGAGATCGAGCGCCGAGGCATGGAGGAGGTGGGCATCTACCGCGTGTCCGGTGTGGCCACGGACATCCAGGC
ACTGAAGGCAGCCTTCGACGTCAAAGCCCTTCAGCGGCCAGTAGCATCTGACTTTGAGCCTCAGGGTCTGAGTGAAGCCGCTCGTTGGAA
CTCCAAGGAAAACCTTCTCGCTGGACCCAGTGAAAATGACCCCAACCTTTTCGTTGCACTGTATGATTTTGTGGCCAGTGGAGATAACAC
TCTAAGCATAACTAAAGGTGAAAAGCTCCGGGTCTTAGGCTATAATCACAATGGGGAATGGTGTGAAGCCCAAACCAAAAATGGCCAAGG
CTGGGTCCCAAGCAACTACATCACGCCAGTCAACAGTCTGGAGAAACACTCCTGGTACCATGGGCCTGTGTCCCGCAATGCCGCTGAGTA
TCTGCTGAGCAGCGGGATCAATGGCAGCTTCTTGGTGCGTGAGAGTGAGAGCAGTCCTGGCCAGAGGTCCATCTCGCTGAGATACGAAGG
GAGGGTGTACCATTACAGGATCAACACTGCTTCTGATGGCAAGCTCTACGTCTCCTCCGAGAGCCGCTTCAACACCCTGGCCGAGTTGGT
TCATCATCATTCAACGGTGGCCGACGGGCTCATCACCACGCTCCATTATCCAGCCCCAAAGCGCAACAAGCCCACTGTCTATGGTGTGTC
CCCCAACTACGACAAGTGGGAGATGGAACGCACGGACATCACCATGAAGCACAAGCTGGGCGGGGGCCAGTACGGGGAGGTGTACGAGGG
CGTGTGGAAGAAATACAGCCTGACGGTGGCCGTGAAGACCTTGAAGGAGGACACCATGGAGGTGGAAGAGTTCTTGAAAGAAGCTGCAGT
CATGAAAGAGATCAAACACCCTAACCTGGTGCAGCTCCTTGGGGTCTGCACCCGGGAGCCCCCGTTCTATATCATCACTGAGTTCATGAC
CTACGGGAACCTCCTGGACTACCTGAGGGAGTGCAACCGGCAGGAGGTGAACGCCGTGGTGCTGCTGTACATGGCCACTCAGATCTCGTC
AGCCATGGAGTACCTGGAGAAGAAAAACTTCATCCACAGAGATCTTGCTGCCCGAAACTGCCTGGTAGGGGAGAACCACTTGGTGAAGGT
AGCTGATTTTGGCCTGAGCAGGTTGATGACAGGGGACACCTACACAGCCCATGCTGGAGCCAAGTTCCCCATCAAATGGACTGCACCCGA
GAGCCTGGCCTACAACAAGTTCTCCATCAAGTCCGACGTCTGGGCATTTGGAGTATTGCTTTGGGAAATTGCTACCTATGGCATGTCCCC
TTACCCGGGAATTGACCTGTCCCAGGTGTATGAGCTGCTAGAGAAGGACTACCGCATGGAGCGCCCAGAAGGCTGCCCAGAGAAGGTCTA
TGAACTCATGCGAGCATGTTGGCAGTGGAATCCCTCTGACCGGCCCTCCTTTGCTGAAATCCACCAAGCCTTTGAAACAATGTTCCAGGA
ATCCAGTATCTCAGACGAAGTGGAAAAGGAGCTGGGGAAACAAGGCGTCCGTGGGGCTGTGAGTACCTTGCTGCAGGCCCCAGAGCTGCC
CACCAAGACGAGGACCTCCAGGAGAGCTGCAGAGCACAGAGACACCACTGACGTGCCTGAGATGCCTCACTCCAAGGGCCAGGGAGAGAG
CGATCCTCTGGACCATGAGCCTGCCGTGTCTCCATTGCTCCCTCGAAAAGAGCGAGGTCCCCCGGAGGGCGGCCTGAATGAAGATGAGCG
CCTTCTCCCCAAAGACAAAAAGACCAACTTGTTCAGCGCCTTGATCAAGAAGAAGAAGAAGACAGCCCCAACCCCTCCCAAACGCAGCAG
CTCCTTCCGGGAGATGGACGGCCAGCCGGAGCGCAGAGGGGCCGGCGAGGAAGAGGGCCGAGACATCAGCAACGGGGCACTGGCTTTCAC
CCCCTTGGACACAGCTGACCCAGCCAAGTCCCCAAAGCCCAGCAATGGGGCTGGGGTCCCCAATGGAGCCCTCCGGGAGTCCGGGGGCTC
AGGCTTCCGGTCTCCCCACCTGTGGAAGAAGTCCAGCACGCTGACCAGCAGCCGCCTAGCCACCGGCGAGGAGGAGGGCGGTGGCAGCTC
CAGCAAGCGCTTCCTGCGCTCTTGCTCCGCCTCCTGCGTTCCCCATGGGGCCAAGGACACGGAGTGGAGGTCAGTCACGCTGCCTCGGGA
CTTGCAGTCCACGGGAAGACAGTTTGACTCGTCCACATTTGGAGGGCACAAAAGTGAGAAGCCGGCTCTGCCTCGGAAGAGGGCAGGGGA
GAACAGGTCTGACCAGGTGACCCGAGGCACAGTAACGCCTCCCCCCAGGCTGGTGAAAAAGAATGAGGAAGCTGCTGATGAGGTCTTCAA
AGACATCATGGAGTCCAGCCCGGGCTCCAGCCCGCCCAACCTGACTCCAAAACCCCTCCGGCGGCAGGTCACCGTGGCCCCTGCCTCGGG
CCTCCCCCACAAGGAAGAAGCTGGAAAGGGCAGTGCCTTAGGGACCCCTGCTGCAGCTGAGCCAGTGACCCCCACCAGCAAAGCAGGCTC
AGGTGCACCAGGGGGCACCAGCAAGGGCCCCGCCGAGGAGTCCAGAGTGAGGAGGCACAAGCACTCCTCTGAGTCGCCAGGGAGGGACAA
GGGGAAATTGTCCAGGCTCAAACCTGCCCCGCCGCCCCCACCAGCAGCCTCTGCAGGGAAGGCTGGAGGAAAGCCCTCGCAGAGCCCGAG
CCAGGAGGCGGCCGGGGAGGCAGTCCTGGGCGCAAAGACAAAAGCCACGAGTCTGGTTGATGCTGTGAACAGTGACGCTGCCAAGCCCAG
CCAGCCGGGAGAGGGCCTCAAAAAGCCCGTGCTCCCGGCCACTCCAAAGCCACAGTCCGCCAAGCCGTCGGGGACCCCCATCAGCCCAGC
CCCCGTTCCCTCCACGTTGCCATCAGCATCCTCGGCCCTGGCAGGGGACCAGCCGTCTTCCACCGCCTTCATCCCTCTCATATCAACCCG
AGTGTCTCTTCGGAAAACCCGCCAGCCTCCAGAGCGGATCGCCAGCGGCGCCATCACCAAGGGCGTGGTCCTGGACAGCACCGAGGCGCT
GTGCCTCGCCATCTCTAGGAACTCCGAGCAGATGGCCAGCCACAGCGCAGTGCTGGAGGCCGGCAAAAACCTCTACACGTTCTGCGTGAG
CTATGTGGATTCCATCCAGCAAATGAGGAACAAGTTTGCCTTCCGAGAGGCCATCAACAAACTGGAGAATAATCTCCGGGAGCTTCAGAT
CTGCCCGGCGACAGCAGGCAGTGGTCCAGCGGCCACTCAGGACTTCAGCAAGCTCCTCAGTTCGGTGAAGGAAATCAGTGACATAGTGCA
GAGGTAGCAGCAGTCAGGGGTCAGGTGTCAGGCCCGTCGGAGCTGCCTGCAGCACATGCGGGCTCGCCCATACCCGTGACAGTGGCTGAC
AAGGGACTAGTGAGTCAGCACCTTGGCCCAGGAGCTCTGCGCCAGGCAGAGCTGAGGGCCCTGTGGAGTCCAGCTCTACTACCTACGTTT
GCACCGCCTGCCCTCCCGCACCTTCCTCCTCCCCGCTCCGTCTCTGTCCTCGAATTTTATCTGTGGAGTTCCTGCTCCGTGGACTGCAGT
CGGCATGCCAGGACCCGCCAGCCCCGCTCCCACCTAGTGCCCCAGACTGAGCTCTCCAGGCCAGGTGGGAACGGCTGATGTGGACTGTCT
TTTTCATTTTTTTCTCTCTGGAGCCCCTCCTCCCCCGGCTGGGCCTCCTTCTTCCACTTCTCCAAGAATGGAAGCCTGAACTGAGGCCTT
GTGTGTCAGGCCCTCTGCCTGCACTCCCTGGCCTTGCCCGTCGTGTGCTGAAGACATGTTTCAAGAACCGCATTTCGGGAAGGGCATGCA
CGGGCATGCACACGGCTGGTCACTCTGCCCTCTGCTGCTGCCCGGGGTGGGGTGCACTCGCCATTTCCTCACGTGCAGGACAGCTCTTGA
TTTGGGTGGAAAACAGGGTGCTAAAGCCAACCAGCCTTTGGGTCCTGGGCAGGTGGGAGCTGAAAAGGATCGAGGCATGGGGCATGTCCT
TTCCATCTGTCCACATCCCCAGAGCCCAGCTCTTGCTCTCTTGTGACGTGCACTGTGAATCCTGGCAAGAAAGCTTGAGTCTCAAGGGTG
GCAGGTCACTGTCACTGCCGACATCCCTCCCCCAGCAGAATGGAGGCAGGGGACAAGGGAGGCAGTGGCTAGTGGGGTGAACAGCTGGTG
CCAAATAGCCCCAGACTGGGCCCAGGCAGGTCTGCAAGGGCCCAGAGTGAACCGTCCTTTCACACATCTGGGTGCCCTGAAAGGGCCCTT
CCCCTCCCCCACTCCTCTAAGACAAAGTAGATTCTTACAAGGCCCTTTCCTTTGGAACAAGACAGCCTTCACTTTTCTGAGTTCTTGAAG
CATTTCAAAGCCCTGCCTCTGTGTAGCCGCCCTGAGAGAGAATAGAGCTGCCACTGGGCACCTGCGCACAGGTGGGAGGAAAGGGCCTGG
CCAGTCCTGGTCCTGGCTGCACTCTTGAACTGGGCGAATGTCTTATTTAATTACCGTGAGTGACATAGCCTCATGTTCTGTGGGGGTCAT
CAGGGAGGGTTAGGAAAACCACAAACGGAGCCCCTGAAAGCCTCACGTATTTCACAGAGCACGCCTGCCATCTTCTCCCCGAGGCTGCCC
CAGGCCGGAGCCCAGATACGGGGGCTGTGACTCTGGGCAGGGACCCGGGGTCTCCTGGACCTTGACAGAGCAGCTAACTCCGAGAGCAGT
GGGCAGGTGGCCGCCCCTGAGGCTTCACGCCGGGAGAAGCCACCTTCCCACCCCTTCATACCGCCTCGTGCCAGCAGCCTCGCACAGGCC
CTAGCTTTACGCTCATCACCTAAACTTGTACTTTATTTTTCTGATAGAAATGGTTTCCTCTGGATCGTTTTATGCGGTTCTTACAGCACA
TCACCTCTTTGCCCCCGACGGCTGTGACGCAGCCGGAGGGAGGCACTAGTCACCGACAGCGGCCTTGAAGACAGAGCAAAGCGCCCACCC
AGGTCCCCCGACTGCCTGTCTCCATGAGGTACTGGTCCCTTCCTTTTGTTAACGTGATGTGCCACTATATTTTACACGTATCTCTTGGTA
TGCATCTTTTATAGACGCTCTTTTCTAAGTGGCGTGTGCATAGCGTCCTGCCCTGCCCCCTCGGGGGCCTGTGGTGGCTCCCCCTCTGCT
TCTCGGGGTCCAGTGCATTTTGTTTCTGTATATGATTCTCTGTGGTTTTTTTTGAATCCAAATCTGTCCTCTGTAGTATTTTTTAAATAA
ATCAGTGTTTACATTAGAA""".replace("\n", "").replace(" ", "")

bp_index = 4073
import signal
import sys

# FIX PRO WINDOWS: Knihovna transformers hledá signál, který Windows nemá
if not hasattr(signal, "SIGALRM"):
    signal.SIGALRM = signal.SIGTERM 

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import pandas as pd
import gc

# 1. Definice zařízení a modelu
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"

print(f"Načítám tokenizer a model na {device}...")

# trust_remote_code=True je nutné pro tento model
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True).to(device).half()
model.eval()

print("Připraveno.")

Načítám tokenizer a model na cuda...
Připraveno.


In [28]:
results = []
# Zvolíme maximální délku, kterou model zvládne (Nucleotide Transformer má limit 8192)
# Použijeme např. 8000 pro jistotu
MAX_WINDOW = 8000 
polomery = list(range(200, min(bp_index, len(full_sequence) - bp_index) + 1, 200))

print(f"Spouštím výpočet s fixním oknem {MAX_WINDOW} bp...")

for polomer in polomery:
    # 1. Definice biologického segmentu
    bio_segment = full_sequence[bp_index - polomer : bp_index + polomer]
    bio_len = len(bio_segment)
    
    # 2. Vytvoření plného řetězce (padding na obou stranách)
    # Celková délka musí být MAX_WINDOW
    pad_total = MAX_WINDOW - bio_len
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    
    # Použijeme 'N' jako neutrální výplň
    full_padded_seq = ("N" * pad_left) + bio_segment + ("N" * pad_right)
    
    # 3. Tokenizace (vždy stejná délka!)
    inputs = tokenizer(full_padded_seq, return_tensors="pt", add_special_tokens=True).to(device)
    
    # 4. Vytvoření ATTENTION MASKY
    # Pozor: Attention maska musí mít stejnou délku jako input_ids
    # 0 = ignorovat (padding), 1 = pozornost (biologická sekvence)
    attention_mask = torch.zeros_like(inputs['input_ids'])
    
    # Vypočítáme, kde v tokenech začíná a končí naše bio_segment
    # Vzhledem k fixní délce je to vždy na stejném místě
    start_idx = pad_left
    end_idx = pad_left + bio_len
    
    # Maskování (pro zjednodušení maskujeme vše, co není 'N')
    # Protože v Nucleotide Transformeru je 'N' speciální token, 
    # můžeme maskovat vše, co je v našem bio_segmentu
    attention_mask[0, pad_left:pad_left + bio_len] = 1
    inputs['attention_mask'] = attention_mask
    
    # 5. Výpočet
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        # BP je vždy přesně uprostřed okna
        token_idx = inputs['input_ids'].shape[1] // 2 
        bp_embedding = outputs.hidden_states[-1][0, token_idx, :].detach().cpu().float()
        
        results.append({
            "polomer": polomer,
            "bio_seq_len": bio_len,
            "embedding": bp_embedding
        })
    
    print(f"Poloměr {polomer:4} | Hotovo")
    del outputs, inputs
    torch.cuda.empty_cache()

Token indices sequence length is longer than the specified maximum sequence length for this model (7671 > 2048). Running this sequence through the model will result in indexing errors


Spouštím výpočet s fixním oknem 8000 bp...
Poloměr  200 | Hotovo
Poloměr  400 | Hotovo
Poloměr  600 | Hotovo
Poloměr  800 | Hotovo
Poloměr 1000 | Hotovo
Poloměr 1200 | Hotovo
Poloměr 1400 | Hotovo
Poloměr 1600 | Hotovo
Poloměr 1800 | Hotovo
Poloměr 2000 | Hotovo
Poloměr 2200 | Hotovo
Poloměr 2400 | Hotovo
Poloměr 2600 | Hotovo
Poloměr 2800 | Hotovo
Poloměr 3000 | Hotovo
Poloměr 3200 | Hotovo
Poloměr 3400 | Hotovo
Poloměr 3600 | Hotovo
Poloměr 3800 | Hotovo
Poloměr 4000 | Hotovo


In [31]:
import torch.nn.functional as F

if not results:
    print("Chyba: Seznam výsledků je prázdný.")
else:
    # Referenční embedding je ten s největším poloměrem (poslední)
    ref_emb = results[-1]["embedding"]
    final_summary = []
    
    for item in results:
        curr_emb = item["embedding"]
        cos_sim = F.cosine_similarity(curr_emb.unsqueeze(0), ref_emb.unsqueeze(0)).item()
        
        # Přidáváme pouze Polomer, Délku a Cosine Similarity
        final_summary.append({
            "Polomer_bp": item["polomer"],
            "Biologicka_Delka": item["bio_seq_len"],
            "Cosine_Similarity": round(cos_sim, 5)
        })

    df_results = pd.DataFrame(final_summary)
    print("\nSrovnání embeddingů:")
    print(df_results.to_string(index=False))
    df_results.to_csv("analyza_final_v2.csv", index=False)


Srovnání embeddingů:
 Polomer_bp  Biologicka_Delka  Cosine_Similarity
        200               400            0.20692
        400               800            0.65333
        600              1200            0.24122
        800              1600            0.19258
       1000              2000            0.63908
       1200              2400            0.30680
       1400              2800            0.20812
       1600              3200            0.68936
       1800              3600            0.28075
       2000              4000            0.20469
       2200              4400            0.67414
       2400              4800            0.20072
       2600              5200            0.19191
       2800              5600            0.98885
       3000              6000            0.20195
       3200              6400            0.19460
       3400              6800            0.99441
       3600              7200            0.20326
       3800              7600            0.1945